In [ ]:
# hide
import numpy as np
import pyquist as pq


def iter_frames(audio, N_H, N_F):
    x = np.asarray(audio.samples).reshape(-1)
    for start in range(0, len(x) - N_F + 1, N_H):
        yield x[start:start + N_F]


def overlap_add(grains, N_H, sample_rate):
    grains = list(grains)
    N_F = len(grains[0])
    out = np.zeros(N_H * (len(grains) - 1) + N_F)
    for k, g in enumerate(grains):
        if len(g) == N_F:
            out[k * N_H:k * N_H + N_F] += g
    return pq.Audio(out.astype(np.float32), sample_rate)


def hann(n):
    return 0.5 * (1 - np.cos(2 * np.pi * np.arange(n) / n))

In [ ]:
# Granular synthesis: chop the sound into overlapping grains, MANIPULATE them,
# and glue them back. Edit `manipulate` to invent your own effect!
def manipulate(grains):
    grains = [g * hann(len(g)) for g in grains]     # smooth each grain's edges
    out = []                                        # shuffle order within blocks
    for i in range(0, len(grains), 100):
        block = grains[i:i + 100]
        np.random.shuffle(block)
        out += block
    return out


N_F = 2048                     # grain size in samples (~46 ms)
N_H = 1024                     # spacing when extracting grains
N_H_out = 1024                 # spacing when reassembling (change to time-stretch!)

audio = pq.Audio.from_file("../assets/audio-trio.wav")
grains = [g for g in iter_frames(audio, N_H, N_F) if len(g) == N_F]
grains = manipulate(grains)
pq.play(overlap_add(grains, N_H_out, audio.sample_rate))